# Qwen3.8-27B on Colab (vLLM + Cloudflare quick tunnel)
Runtime: A100 (40GB). Run the cells top to bottom, then run the printed `scripts/colab-qwen-connect.sh` line on the Mac.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U vllm
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import os, secrets, subprocess, time, urllib.request

MODEL = "Qwen/Qwen3.8-27B"
SERVED_NAME = "Qwen3.8-27B"
# bf16 weights are ~56GB and do not fit an A100 40GB: quantize on load.
# "fp8" (~28GB, default) or "bitsandbytes" (4bit, ~16GB, more KV headroom).
QUANT = "fp8"  # verified on A100 40GB (needs --enforce-eager + small batch, else CUDA-graph profiling OOMs)
MAX_LEN = 65536
API_KEY = secrets.token_urlsafe(32)

vllm = subprocess.Popen(
    ["vllm", "serve", MODEL, "--served-model-name", SERVED_NAME,
     "--quantization", QUANT, "--max-model-len", str(MAX_LEN),
     "--gpu-memory-utilization", "0.90",
     "--max-num-seqs", "4", "--max-num-batched-tokens", "4096", "--enforce-eager",
     "--reasoning-parser", "qwen3",
     "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
     "--api-key", API_KEY, "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

for _ in range(120):  # up to ~20 min for download + load
    if vllm.poll() is not None:
        raise RuntimeError("vllm exited; see /content/vllm.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)
        break
    except Exception:
        time.sleep(10)
else:
    raise TimeoutError("vllm not ready; see /content/vllm.log")
print("vllm ready")

In [ ]:
import re
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print(f"./scripts/colab-qwen-connect.sh {url} {API_KEY}")

In [ ]:
# Keep the session alive and show recent server logs on demand.
!tail -n 20 /content/vllm.log

In [ ]:
# Video input check: a 2s solid-red clip must be described as red.
import base64, json, subprocess, urllib.request
subprocess.run(["ffmpeg","-y","-loglevel","error","-f","lavfi","-i","color=c=red:s=320x240:d=2:r=8",
                "-pix_fmt","yuv420p","/content/red.mp4"], check=True)
b64 = base64.b64encode(open("/content/red.mp4","rb").read()).decode()
body = {"model": SERVED_NAME, "max_tokens": 512,
        "messages": [{"role": "user", "content": [
            {"type": "video_url", "video_url": {"url": "data:video/mp4;base64," + b64}},
            {"type": "text", "text": "What single color fills this video? Answer in one word."}]}]}
req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", json.dumps(body).encode(),
    {"Content-Type": "application/json", "Authorization": "Bearer " + API_KEY})
msg = json.load(urllib.request.urlopen(req, timeout=600))["choices"][0]["message"]
print(msg.get("content"))
assert "red" in (msg.get("content") or "").lower(), "video not recognised"
print("VIDEO OK")